# (Imports)

In [ ]:
%%capture import_io
import sys; sys.path.append('..')
from c101.helpers import *
import c101.helpers
from c101.combinators_101 import *
from functools import reduce
map = c101.helpers.map_

# Interpreters

In [9]:
import operator

def stacky(program, trace=identity):
    'A stack-oriented expression evaluator.'
    def eval(stack, item):
        if isinstance(item, int):
            return stack + [item]
        return stack[:-2] + [item(stack[-2], stack[-1])]
    def parse(word):
        if re.match(r'-?\d+', word):
            return int(word)
        return getattr(operator, word)
    return reduce(trace(eval), map(trace(parse), program.split(" ")), [])

stacky("2 3 mul", trace)

⇢ parse('2')
⇢ ╰╴ 2
⇢ eval([], 2)
⇢ ╰╴ [2]
⇢ parse('3')
⇢ ╰╴ 3
⇢ eval([2], 3)
⇢ ╰╴ [2, 3]
⇢ parse('mul')
⇢ ╰╴ <built-in function mul>
⇢ eval([2, 3], <built-in function mul>)
⇢ ╰╴ [6]


[6]

In [10]:
stacky("2 3 5 add mul", trace)

⇢ parse('2')
⇢ ╰╴ 2
⇢ eval([], 2)
⇢ ╰╴ [2]
⇢ parse('3')
⇢ ╰╴ 3
⇢ eval([2], 3)
⇢ ╰╴ [2, 3]
⇢ parse('5')
⇢ ╰╴ 5
⇢ eval([2, 3], 5)
⇢ ╰╴ [2, 3, 5]
⇢ parse('add')
⇢ ╰╴ <built-in function add>
⇢ eval([2, 3, 5], <built-in function add>)
⇢ ╰╴ [2, 8]
⇢ parse('mul')
⇢ ╰╴ <built-in function mul>
⇢ eval([2, 8], <built-in function mul>)
⇢ ╰╴ [16]


[16]

In [11]:
stacky("33 2 3 add 5 mul gt")

[True]

## Operator Predicates

In [3]:

def binary_op(op: str) -> Optional[Callable[[Any, Any], Any]]:
  'Returns a function for a binary operator by name.'
  return BINARY_OPS.get(op)

BINARY_OPS = {
  '+': lambda a, b: a + b,
  '-': lambda a, b: a - b,
  '*': lambda a, b: a * b,
  '/': lambda a, b: a / b,
  '==': lambda a, b: a == b,
  '!=': lambda a, b: a != b,
  '<':  lambda a, b: a < b,
  '>':  lambda a, b: a > b,
  '<=': lambda a, b: a <= b,
  '>=': lambda a, b: a >= b,
  'and': lambda a, b: a and b,
  'or': lambda a, b: a or b,
}
for n, f in BINARY_OPS.items():
  f.__qualname__ = f"binary_op({n!r})"

binary_op('==') (2, 2)
binary_op('!=') (2, 2)

True

False

In [4]:
# Create a table where `x OP y` is true:
[
  f'{x} {op} {y}'
  for op in ['<', '==', '>']
  for x in (2, 3, 5)
  for y in (2, 3, 5)
  if binary_op(op)(x, y)
]

['2 < 3',
 '2 < 5',
 '3 < 5',
 '2 == 2',
 '3 == 3',
 '5 == 5',
 '3 > 2',
 '5 > 2',
 '5 > 3']

In [5]:
def op_pred(op: str, b: Any) -> Predicate | None:
  'Returns a predicate function given an operator name and a constant.'
  if pred := binary_op(op):
    return lambda a: pred(a, b)
  if op == "not":
    return lambda a: not a
  if op == "~=":
    return re_pred(b)
  if op == "~!":
    return not_(re_pred(b))
  return None

In [6]:
h = op_pred(">", 3)
h(2)
h(5)

False

True

In [7]:
g = op_pred("~=", 'ab+c')
g('')
g('ab')
g('abbbcc')

False

False

True

----
# The End
----